# Bear_BBR Signal - Data Prep

## Import Libs

In [3297]:
import pandas as pd 
import numpy as np
import os
from pandas import DataFrame, Series
import plotly.graph_objects as go

## Read CSV price data and get DataFrame

In [3528]:
PATH = os.getcwd()
FILE = "../output/FE_V2_GBPUSD_15mins_1yr_End_20260311.csv"
df = pd.read_csv(FILE)
df["Date"] = pd.DatetimeIndex(df["Date"], tz="US/Eastern")
df.set_index("Date", inplace=True)
df.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 24580 entries, 2025-03-11 17:15:00-04:00 to 2026-03-11 16:45:00-04:00
Data columns (total 49 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Open           24580 non-null  float64
 1   High           24580 non-null  float64
 2   Low            24580 non-null  float64
 3   Close          24580 non-null  float64
 4   Idx            24580 non-null  int64  
 5   Body           24580 non-null  float64
 6   Range          24580 non-null  float64
 7   UWick          24580 non-null  float64
 8   LWick          24580 non-null  float64
 9   Close_%High    24580 non-null  float64
 10  Open_%High     24580 non-null  float64
 11  Iday_Idx       24580 non-null  int64  
 12  Iday_High      24580 non-null  float64
 13  Iday_Low       24580 non-null  float64
 14  Iday_Range     24580 non-null  float64
 15  Close_%DHigh   24580 non-null  float64
 16  Open_%DHigh    24580 non-null  float64
 17  Yda

## Add extra features

In [3529]:
def get_yra_std(row: Series, df: DataFrame, n: int):
    std = df["YRA_Diff"].at_time("17:15").iloc[row["Day_Idx"]-n+1:row["Day_Idx"]+1].std()
    return std

pd.options.display.max_rows = 100
pd.options.display.max_columns = 100
df["Yday_Range"] = df.eval("Yday_High - Yday_Low")
df["YRA_Diff"] = df.apply(lambda x: x["Yday_Range"] - x["ADR"], axis=1)
df["YRA_STD"] = df.apply(get_yra_std, axis=1, args=[df.copy(), 30])


In [3530]:
bbu_reversal = df
bbu_reversal.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 24580 entries, 2025-03-11 17:15:00-04:00 to 2026-03-11 16:45:00-04:00
Data columns (total 52 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Open           24580 non-null  float64
 1   High           24580 non-null  float64
 2   Low            24580 non-null  float64
 3   Close          24580 non-null  float64
 4   Idx            24580 non-null  int64  
 5   Body           24580 non-null  float64
 6   Range          24580 non-null  float64
 7   UWick          24580 non-null  float64
 8   LWick          24580 non-null  float64
 9   Close_%High    24580 non-null  float64
 10  Open_%High     24580 non-null  float64
 11  Iday_Idx       24580 non-null  int64  
 12  Iday_High      24580 non-null  float64
 13  Iday_Low       24580 non-null  float64
 14  Iday_Range     24580 non-null  float64
 15  Close_%DHigh   24580 non-null  float64
 16  Open_%DHigh    24580 non-null  float64
 17  Yda

# Bear_BBR Signal Analysis

## Bearish BBR Metrics Function

Return a new dataframe with all occrrences of Bear_BBR signal and get additional metrics such as:
- Entry Price
- Max Gain / Loss
- Gain/Loss to:
    - SMA16,
    - SMA32,
    - BB_Lower,
    - SigLow,
    - First_Bullish_Signal (Bull_0)
    - Bull_Engulf
    - Intraday_Low_Reversal (ILR)
- Whether S/R was tested or broken
- Whether other bearish pattern occurred at the same time

In [3531]:

def bearish_bbr_metrics(df: DataFrame, bear_bbr_signal_idx: list):
    """Return a new dataframe with signal metrics"""

    signal = []
    for signal_idx in bear_bbr_signal_idx:
        row = df.iloc[signal_idx]
        s = {}
        START = row.name
        TD = pd.Timedelta(minutes=15)
        s["Date"] = START
        s["Entry_Price"] = row["Close"] # signal price
        
        day = row["Day_Idx"]
        window = df.query(f"Day_Idx == {day}") # only get df data from the same FX day as signal
        before_signal_window = window[:START-TD] # get df data from start of day until just before signal
        signal_window = window[START:] # get df from signal until end of day (EOD)
        
        Max_High = signal_window[START+TD:]["High"].max() # Max high from signal until EOD
        Min_Low = signal_window[START+TD:]["Low"].min() # Min low from signal until EOD
        s["D_Max_Up"] = s["Entry_Price"] - Max_High # Max price gain from signal until EOD
        s["D_Max_Down"] = s["Entry_Price"] - Min_Low # Max price loss from signal until EOD
        D_Close = signal_window.iloc[-1]["Close"] # Closing price at EOD
        s["D_Gain"] = s["Entry_Price"] - D_Close # Gain at EOD from signal 


        if signal_window.query(f"Close > {row["High"]}").empty is False:
            # If price closed above signal price within the day, 
            # get the dataframe between start and failure time
            EXIT = signal_window[START:].query(f"Close > {row["High"]}").iloc[0].name
            entry_window = signal_window[START:EXIT]
        else:
            # If price didn't close above signal price within the day
            # get the dataframe from signal time until EOD
            entry_window = signal_window

        # Did the price go higher than the signal candle high
        if entry_window[START+TD:].query(f"High >= {row["High"]}").empty is False:
            s["Stop_Pips"] = s["Entry_Price"] - row["High"]
        else:
            s["Stop_Pips"] = 0
        
        s["Max_Up"] = s["Entry_Price"] - entry_window[START+TD:]["Close"].max()
        s["Max_Down"] = s["Entry_Price"] - entry_window[START+TD:]["Low"].min()
        

        # If the price fell to the SMA16 after the signal
        # calcuate the gain
        below_sma16 = entry_window.query("Low < SMA16")
        if below_sma16.empty is False:
            s["To_SMA16"] = s["Entry_Price"] - below_sma16.iloc[0]["SMA16"] 
        else: 
            s["To_SMA16"] = 0

        # If the price fell to the SMA32 after the signal
        # calcuate the gain
        below_sma32 = entry_window.query("Low < SMA32")
        if below_sma32.empty is False:
            s["To_SMA32"] = s["Entry_Price"] - below_sma32.iloc[0]["SMA32"]
        else:
            s["To_SMA32"] = 0

        # If the price fell to the BB_Lower_16_2 after the signal
        # calcuate the gain
        below_bbl = entry_window.query("Low < BB_Lower_16_2")
        if below_bbl.empty is False:
            s["To_BBL_16_2"] = s["Entry_Price"] - below_bbl.iloc[0]["BB_Lower_16_2"]
        else:
            s["To_BBL_16_2"] = 0

        # If the price fell to the Sig_Low after the signal
        # calcuate the gain
        below_sig_low = entry_window.query("Low < Sig_Low")
        if below_sig_low.empty is False:
            s["To_Sig_Low"] = s["Entry_Price"] - below_sig_low.iloc[0]["Sig_Low"]
        else:
            s["To_Sig_Low"] = 0

        below_sr = entry_window.query("S_R > 3")["Sig_Low"].unique()
        for i in range(len(below_sr)):
            s[f"To_S_{i}"] = s["Entry_Price"] - below_sr[i]


        # Check if bullish signals occured during winddow
        NULL_Timestamp = pd.Timestamp(0, tz="US/Eastern")
        to_bull_bbr = entry_window.query("Bull_BBR == True")
        if to_bull_bbr.empty is False:
            s["To_Bull_BBR"] = s["Entry_Price"] - to_bull_bbr.iloc[0]["Close"]
            to_bull_bbr_time_stamp = to_bull_bbr.iloc[0].name
        else:
            s["To_Bull_BBR"] = 0
            to_bull_bbr_time_stamp = NULL_Timestamp

        to_bull_engulf = entry_window.query("Bull_Engulf == True")
        if to_bull_engulf.empty is False:
            s["To_Bull_Engulf"] = s["Entry_Price"] - to_bull_engulf.iloc[0]["Close"]
            to_bull_engulf_time_stamp = to_bull_engulf.iloc[0].name
        else:
            s["To_Bull_Engulf"] = 0
            to_bull_engulf_time_stamp = NULL_Timestamp

        to_ilr = entry_window.query("ILR == True")
        if to_ilr.empty is False:
            s["To_ILR"] = s["Entry_Price"] - to_ilr.iloc[0]["Close"]
            to_ilr_time_stamp = to_ilr.iloc[0].name
        else:
            s["To_ILR"] = 0
            to_ilr_time_stamp = NULL_Timestamp

        first_bullish_signal = min([to_bull_bbr_time_stamp, to_bull_engulf_time_stamp, to_ilr_time_stamp])
        if first_bullish_signal > NULL_Timestamp:
            s["To_Bull_0"] = s["Entry_Price"] - entry_window.loc[first_bullish_signal]["Close"]
        else:
            s["To_Bull_0"] = 0

        # how man times was resistance tested before signal
        r_test = before_signal_window.query("S_R == 1")["S_R"].count() 
        # how man times was resistance broken before signal
        r_break = before_signal_window.query("S_R == 2")["S_R"].count()
        # how man times was support tested before signal
        s_test = before_signal_window.query("S_R == 3")["S_R"].count()
        # how man times was support broken before signal
        s_break = before_signal_window.query("S_R == 4")["S_R"].count()

        h1_range = window[:START].iloc[-4:]["High"].max() - window[:START].iloc[-4:]["Low"].min()


        # append other useful df metrics to new df
        s["Range"] = row["Range"]
        s["ATR"] = row["ATR"]
        s["Close_%High"] = row["Close_%High"]
        s["SMA_Trend"] = row["SMA_Trend"]
        s["SMA16_Slope"] = row["SMA16_Slope"]
        s["SMA32_Slope"] = row["SMA32_Slope"]
        s["RSI"] = row["RSI"]
        s["IHR"] = row["IHR"]
        s["Dark_Cloud"] = row["Dark_Cloud"]
        s["Shooting_Star"] = row["Shooting_Star"]
        s["Bear_Engulf"] = row["Bear_Engulf"]
        s["Iday_Range"] = row["Iday_Range"]
        s["ADR"] = row["ADR"]
        s["Close_%DHigh"] = row["Close_%DHigh"]
        s["SMA16"] = row["SMA16"]
        s["SMA32"] = row["SMA32"]
        s["S_R"] = row["S_R"]
        s["YRA_Diff"] = row["YRA_Diff"]
        s["YRA_STD"] = row["YRA_STD"]
        s["R_Tested"] = r_test
        s["R_Broken"] = r_break
        s["S_Tested"] = s_test
        s["S_Broken"] = s_break
        s["H1_Range"] = h1_range
        

        signal.append(s)
    sdf = pd.DataFrame(signal)
    return sdf


### Get Bear_BBR Metrics

In [3532]:
signal_idx_list = bbu_reversal.query("Bear_BBR == True")["Idx"].to_list()
bearish_bbr_df = bearish_bbr_metrics(bbu_reversal, signal_idx_list)
bearish_bbr_df.tail(10)

,Date,Entry_Price,D_Max_Up,D_Max_Down,D_Gain,Stop_Pips,Max_Up,Max_Down,To_SMA16,To_SMA32,To_BBL_16_2,To_Sig_Low,To_Bull_BBR,To_Bull_Engulf,To_ILR,To_Bull_0,Range,ATR,Close_%High,SMA_Trend,SMA16_Slope,SMA32_Slope,RSI,IHR,Dark_Cloud,Shooting_Star,Bear_Engulf,Iday_Range,ADR,Close_%DHigh,SMA16,SMA32,S_R,YRA_Diff,YRA_STD,R_Tested,R_Broken,S_Tested,S_Broken,H1_Range,To_S_0,To_S_1,To_S_2
379,2026-03-04 20:30:00-05:00,1.336975,-0.000705,0.007225,0.001090,0.000000,-0.000260,0.007225,-0.000480,-0.000002,0.000881,0.00050,0.00254,0.000495,0.00048,0.00048,0.001820,0.000709,0.942308,2,5.651495,28.118933,48.294997,NaN,NaN,NaN,NaN,0.002235,0.010132,0.776286,1.337455,1.336977,NaN,-0.000132,0.003869,0,0,0,0,0.001840,0.00050,0.00638,NaN
380,2026-03-05 20:15:00-05:00,1.335575,-0.006055,0.004420,-0.004505,-0.001080,-0.001145,0.000000,-0.000215,0.000000,0.000000,0.00000,0.00000,0.000000,0.00000,0.00000,0.001195,0.000619,0.903766,2,5.828753,40.948449,53.285765,NaN,NaN,NaN,NaN,0.002125,0.010079,0.541176,1.335790,1.334766,NaN,-0.001119,0.003272,0,0,0,0,0.001265,NaN,NaN,NaN
381,2026-03-06 08:45:00-05:00,1.333850,-0.007780,0.001320,-0.006230,-0.003810,-0.004130,0.001320,0.000277,-0.000815,0.000000,-0.00075,0.00000,0.000000,0.00000,0.00000,0.004115,0.001975,0.925881,1,52.073538,-4.111065,48.065741,NaN,NaN,NaN,NaN,0.007940,0.010079,0.660579,1.333573,1.334665,4.0,-0.001119,0.003272,2,2,0,2,0.006520,-0.00075,NaN,NaN
382,2026-03-06 10:15:00-05:00,1.335440,-0.006190,0.000500,-0.004640,-0.002685,-0.003570,0.000500,0.000000,0.000000,0.000000,0.00000,0.00000,0.000000,0.00000,0.00000,0.002985,0.002402,0.899497,1,45.119118,-0.179049,52.486291,NaN,True,NaN,NaN,0.007940,0.010079,0.460327,1.334037,1.334415,1.0,-0.001119,0.003272,3,2,0,3,0.005245,NaN,NaN,NaN
383,2026-03-09 01:30:00-04:00,1.332340,-0.012375,0.002085,-0.011050,-0.001190,-0.001480,0.001720,0.001164,0.000000,0.000000,0.00000,0.00000,0.000000,0.00000,0.00000,0.002000,0.001265,0.595000,1,55.446148,-24.326933,54.360800,NaN,NaN,NaN,NaN,0.006865,0.009884,0.412964,1.330229,1.330614,NaN,0.000591,0.003321,0,0,0,0,0.003950,NaN,NaN,NaN
384,2026-03-09 02:00:00-04:00,1.332730,-0.011985,0.002475,-0.010660,-0.001345,-0.001735,0.002110,0.001554,0.000000,0.000000,0.00000,0.00000,0.000000,0.00000,0.00000,0.001590,0.001371,0.845912,2,59.302734,0.835504,55.186083,NaN,True,NaN,NaN,0.006865,0.009884,0.356154,1.330734,1.330619,NaN,0.000591,0.003321,0,0,0,0,0.003940,NaN,NaN,NaN
385,2026-03-09 11:15:00-04:00,1.339030,-0.005685,0.000830,-0.004360,-0.000750,-0.001385,0.000680,0.000000,0.000000,0.000000,0.00000,0.00000,0.000000,0.00000,0.00000,0.001240,0.001635,0.604839,2,60.917219,56.519957,65.245502,NaN,NaN,NaN,NaN,0.011760,0.009884,0.088435,1.336594,1.335089,NaN,0.000591,0.003321,2,3,2,2,0.004560,NaN,NaN,NaN
386,2026-03-09 15:45:00-04:00,1.342940,-0.001775,0.000530,-0.000450,-0.001630,-0.000935,0.000530,0.000000,0.000000,0.000000,0.00000,0.00000,0.000000,0.00000,0.00000,0.001690,0.001246,0.964497,2,60.689993,61.532578,68.540948,NaN,NaN,NaN,NaN,0.016260,0.009884,0.100246,1.339856,1.338511,NaN,0.000591,0.003321,2,4,2,2,0.006205,NaN,NaN,NaN
387,2026-03-10 02:45:00-04:00,1.345030,-0.003310,0.003805,0.003225,-0.000745,-0.002365,0.000320,0.000000,0.000000,0.000000,0.00000,0.00000,0.000000,0.00000,0.00000,0.001135,0.000777,0.656388,1,49.774958,26.133723,63.597555,NaN,True,NaN,NaN,0.004415,0.010196,0.168743,1.343000,1.343020,3.0,0.006209,0.003365,0,2,2,1,0.002185,NaN,NaN,NaN
388,2026-03-10 20:45:00-04:00,1.343060,-0.002700,0.003715,0.001875,-0.000690,-0.001175,0.000420,0.000000,0.000187,0.000000,0.00000,0.00000,0.000000,0.00000,0.00000,0.000775,0.000553,0.890323,1,29.403071,-26.588919,51.484603,NaN,NaN,NaN,NaN,0.002735,0.009747,0.252285,1.342066,1.342959,NaN,-0.002632,0.003298,0,1,0,0,0.002275,NaN,NaN,NaN


## Signal Stats

### Initial Signal Stats

In [3533]:
# get the total number of bear_bbr signals
total_signals_count = bearish_bbr_df[bearish_bbr_df.columns[0]].count()

day_closed_lower = bearish_bbr_df.query("D_Gain > 0")
day_closed_higher = bearish_bbr_df.query("D_Gain <= 0")
day_closed_lower_count = day_closed_lower["D_Gain"].count()
day_closed_higher_count = day_closed_higher["D_Gain"].count()
day_closed_lower_pct = round(day_closed_lower_count / total_signals_count * 100,2)
day_closed_lower_pips = bearish_bbr_df["D_Gain"].sum().round(6)

# filter the dataframe for occurences where after the signal
# the price moved lower more than it moved higher
d_max_down_greater = bearish_bbr_df.query("D_Max_Up.abs() < D_Max_Down")
# get the count of occurences where price dropped more then it rose after signal
d_max_down_greater_count = d_max_down_greater[d_max_down_greater.columns[0]].count()

# percentage of where price fell further than the max it went against the entry price
d_max_down_greater_pct = round(d_max_down_greater_count / total_signals_count * 100, 2)

max_down_win = bearish_bbr_df.query("Max_Down > Max_Up.abs()")["Date"].count()
max_down_loss = bearish_bbr_df.query("Max_Down <= Max_Up.abs()")["Date"].count()
total_max_down_trades = max_down_win + max_down_loss
max_down_win_rate = round((max_down_win / total_max_down_trades) * 100 ,2)

max_down_pips = bearish_bbr_df["Max_Down"].sum().round(6)
max_up_pips = bearish_bbr_df["Max_Up"].sum().round(6)
total_max_down_gain = max_down_pips + max_up_pips

average_max_up = bearish_bbr_df["Max_Up"].mean().round(6)
average_max_down = bearish_bbr_df["Max_Down"].mean().round(6)


print(
    f"""
    ========= Initial Bear_BBR Statistics =============

    Total Bear_BBR Signals: {total_signals_count}

    ** Where the move down from the signal resulted **
    ** in a positive gain at the close of the day **
    Day Closed Lower (Count): {day_closed_lower_count}
    Day Closed Higher (Count): {day_closed_higher_count}
    Day Closed Lower (%): {day_closed_lower_pct}
    Day Closed Lower (Pips): {day_closed_lower_pips}

    ** Where after the signal the max move down on the day **
    ** was greater than the max move up **
    D_Max_Down > D_Max_Up (Count): {d_max_down_greater_count}
    D_Max_Down > D_Max_Up (%): {d_max_down_greater_pct}

    ** Where the max move down after the signal **
    ** was greater than the max move up (exit if price closes above signal high) **
    Max_Down > Max_Up (Count): {max_down_win}
    Max_Down <= Max_Up (Count): {max_down_loss}
    Total Trades (Max_Down): {total_max_down_trades}
    Max_Down > Max_Up (Win Rate): {max_down_win_rate}

    Max_Down Pips: {max_down_pips}
    Max_Up Pips: {max_up_pips}
    Total Max_Down Gain: {total_max_down_gain}
    
    Average Max_Up Pips: {average_max_up}
    Average Max_Down Pips: {average_max_down}
 
    ==============================================================================
    """
)


    ========= Initial Bear_BBR Statistics =============

    Total Bear_BBR Signals: 389

    ** Where the move down from the signal resulted **
    ** in a positive gain at the close of the day **
    Day Closed Lower (Count): 194
    Day Closed Higher (Count): 195
    Day Closed Lower (%): 49.87
    Day Closed Lower (Pips): -0.077085

    ** Where after the signal the max move down on the day **
    ** was greater than the max move up **
    D_Max_Down > D_Max_Up (Count): 189
    D_Max_Down > D_Max_Up (%): 48.59

    ** Where the max move down after the signal **
    ** was greater than the max move up (exit if price closes above signal high) **
    Max_Down > Max_Up (Count): 189
    Max_Down <= Max_Up (Count): 196
    Total Trades (Max_Down): 385
    Max_Down > Max_Up (Win Rate): 49.09

    Max_Down Pips: 0.78982
    Max_Up Pips: -0.364475
    Total Max_Down Gain: 0.425345
    
    Average Max_Up Pips: -0.000947
    Average Max_Down Pips: 0.002051
 
    


### Summary

- From the initial numbers, we can see that the Bear_BBR signal occurred `149` times.
- After the signal occuring, the *max price move down* was greater than the *max price move up* on the day, `47.65%` of the time.
- When the signal occurred, `44.59%` of the time the price dropped more than it gained after the signal, before closing above the signal high.
- The total gains where the price fell more than it rose after the signal (before closing above the signal high) was `1362` pips.
- The average number of pips where price moved against the signal was `10.56` pips
- The average number of pips gained before price closed above the signal high was `19.76` pips.

#### What does this tell us?

- We can see just under 50% of the time, the price moves lower after the signal, *before or without closing above the signal high*.
- Additionally, the gains from when the price moves in favour of the signal is greater than when the price moves against the signal. In short the signal generates more gains than losses overall.
- On average, the price moves down nearly double the amount it moves against the signal.

#### What information is missing?

- Although the signal gains more pips overall, this doesn't tell us the best place to take profit to capture the max move down.
- It is not clear what factors make a move down more likely, or when the signal is more likely to fail.


## Investigating factors that increase signal accuracy

### Holding a short position from signal entry to close of the day



In [3534]:
def daily_pip_gain(df: Series, pct_adr: int):
    d_pip_gain = df["D_Gain"] if abs(df["D_Max_Up"]) < (df["ADR"] * pct_adr) else (df["ADR"] * -pct_adr)
    return d_pip_gain

def condition_stats(df: DataFrame, pct_adr: int):

    df["Pip_Gain"] = df.apply(daily_pip_gain, axis=1, args=[pct_adr])
    df["ADR%"] = df.apply(lambda x: x["ADR"] * pct_adr, axis=1)
    
    success_series = df.query(f"D_Max_Up.abs() < (ADR * {pct_adr})")["Pip_Gain"]
    fail_series = df.query(f"D_Max_Up.abs() >= (ADR * {pct_adr})")["Pip_Gain"]

    success_total = success_series.count()
    fail_total = fail_series.count()
    total_trades = success_total + fail_total
    
    win_rate = round(success_total / total_trades * 100,2)
    condition_total_pips = round(success_series.sum() + fail_series.sum(),6)

    return {
        "Pct_ADR": pct_adr,
        "Pct_ADR_Avg": df["ADR%"].mean(),
        "ADR_Avg": df["ADR"].mean(),
        "Total Trades": total_trades,
        "Win": success_total,
        "Loss": fail_total,
        "Win Rate": win_rate,
        "Avg_win": success_series.mean(),
        "Avg_Loss": fail_series.mean(),
        "Total Pips": condition_total_pips
    }
    
# columns
cols1 = [
    "Date", "Entry_Price", "Pip_Gain", "D_Max_Up", "ADR%", "D_Gain", "ADR"]
#df
d_gain_df = bearish_bbr_df


#### Using a stop loss set to a percentage of the ADR

Since we already know from the intial stats that the price will close lower on the day 48% of the time after the signal, we should try to limit the losses for the 52% of the times where it fails.

A `stop loss` is used to limit losses, and can be static (e.g. 10 pips) or dynamic (e.g. set as multiple of the `ATR`, or a trailing stop ... etc.).
The ATR takes into account the volatility of the last *x* candles, so is usually a good choice. 

For this particular scenario however, when the `Bear_BBR` signal occurs we are looking to hold a short position until the close of the day. The ATR is not the best approach here, because ATR will look at the average candle range within increments of the timeframe period (e.g. 15min) over *x* period of candles. 

To address this I will use a percentage of the the average daily range (**ADR**) as a stop loss. The ADR calculates the average daily price movement over last 30 days of trading, which will tell me the average number of pips the price moves within a day over that period.

To find a reasonable stop loss range, I'll iterate through different ADR percentages from 1 to 100.

In [3535]:
d_gain_experimental_sl = [condition_stats(d_gain_df, x * 0.01) for x in range(1,101)]
d_gain_ex_sl_res_df = pd.DataFrame(d_gain_experimental_sl)
d_gain_ex_sl_res_df

,Pct_ADR,Pct_ADR_Avg,ADR_Avg,Total Trades,Win,Loss,Win Rate,Avg_win,Avg_Loss,Total Pips
0,0.01,0.000090,0.008968,385,14,371,3.64,0.003741,-0.000090,0.019091
1,0.02,0.000179,0.008968,385,21,364,5.45,0.003701,-0.000179,0.012389
2,0.03,0.000269,0.008968,385,35,350,9.09,0.003544,-0.000269,0.030005
3,0.04,0.000359,0.008968,385,41,344,10.65,0.003540,-0.000358,0.021859
4,0.05,0.000448,0.008968,385,47,338,12.21,0.003633,-0.000449,0.019088
5,0.06,0.000538,0.008968,385,52,333,13.51,0.003504,-0.000539,0.002702
6,0.07,0.000628,0.008968,385,58,327,15.06,0.003487,-0.000628,-0.003162
7,0.08,0.000717,0.008968,385,66,319,17.14,0.003398,-0.000717,-0.004542
8,0.09,0.000807,0.008968,385,73,312,18.96,0.003366,-0.000805,-0.005441
9,0.10,0.000897,0.008968,385,80,305,20.78,0.003419,-0.000896,0.000248


- This shows the stop_loss is directly related to the win rate.
- It also shows that even with a higher win rate, it does not mean a higher `total pips` value. This makes sense because the larger the stop loss, the larger each loss will be.
- The ideal stop loss will be as low as possible and yield a reasonably high `total pips`.
- The maximum total pips returned was 0.011257 which was a result of 3% ADR stop loss, however we can't take this at face value because it could be an outlier. We also need to consider that when a trade is placed, a percentage of equity will be risked (e.g. 1%), so a lower ADR percentage will allow for a larger trade size to be used, which maximises the monetary return.
- We should look at the top ten `total pips` values and see what range of ADR percentages provided the best return.

In [3536]:
top_ten_returns_by_adr_pct = d_gain_ex_sl_res_df["Total Pips"].nlargest(10)
top_ten_returns_by_adr_pct

27    0.039018
15    0.037360
16    0.033129
17    0.031198
2     0.030005
47    0.023552
14    0.022850
45    0.022004
28    0.021971
3     0.021859
Name: Total Pips, dtype: float64

The output shows the top ten `total pips` values by their index. Since the percent of ADR value is the `index+1`, we can see what contiguous index values provide the best return. I prefer contiguous values to get a better approximation of ADR percent ranges that work well, which should help to avoid outliers that may have been a fluke. 

From the output there's 3 contiguous index values between 15-17 which map to 16-18% ADR.

When you look at 16-18% ADR, the win rate is (33.59, 37.50, 39.06) but the average win for 16% ADR (0.003070) is double the average loss (0.001437). The 16% ADR value also yields the second highest `total pips` return out of all ADR percentage values. So 16% ADR looks good becauses it keeps the pip loss small, and still has a reasonable win rate which we can try to improve.

The reason for choosing the 2nd lowest ADR percentage value is that is allows us to take a larger trade size while risking the same equity percentage. A win with a larger trade value will generate a larger monetary return for the total pips. The reason for discounting 3% ADR is because of the terrible win rate, and the fact it is likely to be specifically good for this current dataset. 

To check the potential of monetary rerturns using this strategy, I'll simulate an account size of $100,000 and a trade risk value of 1%. 

In [3537]:
risk_ex_df = d_gain_ex_sl_res_df
risk_ex_df["Account_Size"] = risk_ex_df.apply(lambda x: 100000, axis=1)
risk_ex_df["Risk"] = risk_ex_df.apply(lambda x: 0.01, axis=1)
risk_ex_df["Trade_Value"] = risk_ex_df.apply(lambda x: round((x["Account_Size"] * x["Risk"])/x["Pct_ADR_Avg"],2), axis=1)
risk_ex_df["Return"] = risk_ex_df.apply(lambda x: round(x["Trade_Value"] * x["Total Pips"],2), axis=1)

risk_ex_df.filter(risk_ex_df["Return"].nlargest(10).index, axis=0)

,Pct_ADR,Pct_ADR_Avg,ADR_Avg,Total Trades,Win,Loss,Win Rate,Avg_win,Avg_Loss,Total Pips,Account_Size,Risk,Trade_Value,Return
0,0.01,0.000090,0.008968,385,14,371,3.64,0.003741,-0.000090,0.019091,100000,0.01,11151202.57,212887.61
2,0.03,0.000269,0.008968,385,35,350,9.09,0.003544,-0.000269,0.030005,100000,0.01,3717067.52,111530.61
1,0.02,0.000179,0.008968,385,21,364,5.45,0.003701,-0.000179,0.012389,100000,0.01,5575601.28,69076.12
3,0.04,0.000359,0.008968,385,41,344,10.65,0.003540,-0.000358,0.021859,100000,0.01,2787800.64,60938.53
4,0.05,0.000448,0.008968,385,47,338,12.21,0.003633,-0.000449,0.019088,100000,0.01,2230240.51,42570.83
15,0.16,0.001435,0.008968,385,126,259,32.73,0.003239,-0.001431,0.037360,100000,0.01,696950.16,26038.06
16,0.17,0.001524,0.008968,385,133,252,34.55,0.003132,-0.001521,0.033129,100000,0.01,655953.09,21731.07
17,0.18,0.001614,0.008968,385,139,246,36.10,0.003078,-0.001612,0.031198,100000,0.01,619511.25,19327.51
14,0.15,0.001345,0.008968,385,117,268,30.39,0.003270,-0.001342,0.022850,100000,0.01,743413.50,16987.00
11,0.12,0.001076,0.008968,385,97,288,25.19,0.003365,-0.001075,0.016760,100000,0.01,929266.88,15574.51


The results show an ADR percentage of 1% yields the highest return but has a 3% win rate. This is clearly an anomaly because 2 to 3% ADR would also yield high returns if this was a good area for a stop loss. 

The better indicator is the range between 16-18%, which is a countiguous range of values with the highest monetary return out of the distribution. Based on this I'll view 16% ADR as an ideal stop loss and look for ways to improve the win rate.

#### Testing if the slope angle increases win rate

Considering the Bear_BBR is a short signal, we want to avoid cases were there is a strong uptrend. There's probably many ways to define a strong uptrend, but I think looking at the angle of the simple moving average (SMA) slopes is a way to define it. Generally, a strong trend or up-move will have SMA's pointing up. To quantify "pointing up" we'll say if the angle is greater than *x* the trend is too bullish, so skip the signal.

I think setting the angle limit to 45 degrees makes sense. Let's see what this shows:

In [3538]:
# conditions
slope_lt_45_df = bearish_bbr_df.query("SMA32_Slope < 45").copy()
slope_lt_45_res = condition_stats(slope_lt_45_df, 0.16)
pd.DataFrame([slope_lt_45_res])

,Pct_ADR,Pct_ADR_Avg,ADR_Avg,Total Trades,Win,Loss,Win Rate,Avg_win,Avg_Loss,Total Pips
0,0.16,0.001436,0.008973,349,116,233,33.24,0.003309,-0.001432,0.050175


This shows a slight reduction in the total number of trades (98), which we'd expect with this filter. We also see a similar win rate, and an increase of pips gained, making the `total pips` 175. It definitely looks better, but we can also test for a range of angles betweem 90 and 0, to see of there is anything better.

In [3539]:
slope_angle_ex = [condition_stats(bearish_bbr_df.query(f"SMA32_Slope < {x}").copy(), 0.16) for x in range(0,91)]
slope_angle_ex_df = pd.DataFrame(slope_angle_ex)
slope_angle_ex_df

,Pct_ADR,Pct_ADR_Avg,ADR_Avg,Total Trades,Win,Loss,Win Rate,Avg_win,Avg_Loss,Total Pips
0,0.16,0.001435,0.008970,91,22,69,24.18,0.002728,-0.001440,-0.039370
1,0.16,0.001435,0.008968,97,24,73,24.74,0.002945,-0.001442,-0.034588
2,0.16,0.001429,0.008934,102,28,74,27.45,0.003218,-0.001439,-0.016368
3,0.16,0.001428,0.008922,108,31,77,28.70,0.003202,-0.001431,-0.010905
4,0.16,0.001424,0.008899,113,34,79,30.09,0.003279,-0.001428,-0.001299
5,0.16,0.001423,0.008891,116,35,81,30.17,0.003292,-0.001424,-0.000138
6,0.16,0.001428,0.008925,118,36,82,30.51,0.003181,-0.001425,-0.002373
7,0.16,0.001425,0.008905,123,38,85,30.89,0.003128,-0.001421,-0.001916
8,0.16,0.001430,0.008939,127,41,86,32.28,0.003145,-0.001421,0.006714
9,0.16,0.001425,0.008909,138,45,93,32.61,0.003354,-0.001417,0.019190


We can view the angles with the largest returns:

In [3540]:
slope_angle_ex_df["Account_Size"] = slope_angle_ex_df.apply(lambda x: 100000, axis=1)
slope_angle_ex_df["Risk"] = slope_angle_ex_df.apply(lambda x: 0.01, axis=1)
slope_angle_ex_df["Trade_Value"] = slope_angle_ex_df.apply(lambda x: round((x["Account_Size"] * x["Risk"])/x["Pct_ADR_Avg"],2), axis=1)
slope_angle_ex_df["Return"] = slope_angle_ex_df.apply(lambda x: round(x["Trade_Value"] * x["Total Pips"],2), axis=1)
slope_angle_ex_df.filter(slope_angle_ex_df["Total Pips"].nlargest(10).index, axis=0)

,Pct_ADR,Pct_ADR_Avg,ADR_Avg,Total Trades,Win,Loss,Win Rate,Avg_win,Avg_Loss,Total Pips,Account_Size,Risk,Trade_Value,Return
33,0.16,0.001429,0.008932,298,103,195,34.56,0.003369,-0.001426,0.068844,100000,0.01,699746.92,48173.38
40,0.16,0.001433,0.008959,335,113,222,33.73,0.003352,-0.001428,0.061773,100000,0.01,697606.38,43093.24
34,0.16,0.001429,0.008933,306,105,201,34.31,0.003315,-0.001425,0.061524,100000,0.01,699664.12,43046.14
31,0.16,0.001431,0.008942,287,99,188,34.49,0.003327,-0.001427,0.061025,100000,0.01,698914.01,42651.23
30,0.16,0.001429,0.008931,272,93,179,34.19,0.003389,-0.001422,0.060630,100000,0.01,699820.19,42430.10
35,0.16,0.001430,0.008936,310,106,204,34.19,0.003318,-0.001427,0.060605,100000,0.01,699410.33,42387.76
49,0.16,0.001435,0.008969,359,121,238,33.70,0.003318,-0.001432,0.060599,100000,0.01,696827.73,42227.06
41,0.16,0.001434,0.008963,336,113,223,33.63,0.003352,-0.001429,0.060160,100000,0.01,697349.27,41952.53
32,0.16,0.001430,0.008936,293,101,192,34.47,0.003293,-0.001427,0.058564,100000,0.01,699421.52,40960.92
48,0.16,0.001435,0.008971,358,120,238,33.52,0.003314,-0.001432,0.056799,100000,0.01,696661.80,39569.69


This shows angles between the range of 28 and 30 have returns above 287 pips, however the best fit is **33** degrees. From this sample, if we only took `Bear_BBR` trades with an SMA32 slope under 29 degrees the `total pips` would be 319.7 with a monetary return of $22575.84 (excluding fees and commissions). 

The results look good, but this only shows filtering for trades which are held until the close of the day. It relies on the day being a complete reversal day and also using a tighter stop loss (16% ADR). These two factors may be unique to this dataset so we need to test over a different year and see if the results are similar.

When I run the tests over the 2025 period with file `FE_V2_GBPUSD_15mins_1yr_End_20250311`, it turns out the optimum stop loss is 2% ADR which gives a 3.64% win rate, but does deliver the highest return (~12k). Obviously this is unrealistic as we have to account for the spread + commissions, so the actual return is probably far less. Outside of that, if we look at the top ten highest `total pips`, the next best stop loss is 29% ADR, which is more reasonable and carries a 50% win rate and 280 pip gain over 147 trades (when sma32_slope < 50).

So comparing the two we can consider:
- The approach to calculating the stop loss is biased to the dataset, which is why the best stop loss (by gains) differs between 2024-2025 and 2025-2026. When looking at common ADR% stop loss values across both data sets, 25-30 seems to appear in both, so would be a good realistic start.
- The optimal slop value also changes, with the optimal slop being under 5 degress in 2024-2025 dataset. The slop is considered in the signal definition as well, so I think this reading is probably biased to the dataset.
- The max move down after the signal was greater than the max move up 47% of the time on both datasets. This shows the signals are good and are correctly identifying a drop in price at least half the time.

Since bollinger bands can be used for breakouts and mean reversion, it does explain why there's ~50% accuracy. In FX there are a lot of days where price moves in a range, or has low trend movement. This means even though a signal predicts a successful fall in price, the price could still reverse and close above the signal candle. To solve this we need to define a take profit target that will maximise the gains on a successful signal. When I briefly investigated strategies for targets by manual chart analysis, there appears to be a relationship between the high of the signal candle and the intraday range. I will delve into this further in the next section.

## High vs ADR% Stop Loss - 38.2% ADR Target

In [3541]:
def get_bear_bbr_pip_gain(
        df: Series, 
        high: Series, 
        low: Series,
        close: Series  
        ):
    """Get the pip gain and apply to df"""

    high_win = 0
    high_loss = 0
    adr_win = 0
    adr_loss = 0
    high_target_pips = 0 
    adr_target_pips = 0 
    high_sl_pips = 0
    adr_sl_pips = 0
    high_gain = 0
    adr_gain = 0
    high_trade_start =  None
    high_trade_end = None
    adr_trade_start =  None
    adr_trade_end = None
    
    if df["Bear_BBR"] is True:
        idx = int(df["Idx"])
        iday_idx = int(df["Iday_Idx"])
        # get signal start of fx day timestamp
        day_start_ts = high.iloc[idx-iday_idx:idx-iday_idx+1].index[0]
        OFFSET = day_start_ts + pd.DateOffset(days=1) # end of signal's fx day
        START = df.name # signal start time
        TD = pd.Timedelta(minutes=15)
        END = OFFSET - TD # 17:00 on the signals fx day

        # get stop loss time:
        sl_window = high.loc[START+TD:END] # from signal idx+1 to EOD
        close_window = close.loc[START+TD:END] # from signal idx+1 to EOD
        high_stop = False 
        high_sl_ts = None # get timestamp when stopped out
        adr_stop = False # update if stopped 
        adr_sl_ts = None # get timestamp when stopped out
        stoploss_high = max(high.iloc[idx-1], high.iloc[idx])

        for i in range(len(sl_window)):
            if sl_window.iloc[i] >= stoploss_high : #stoploss_high:
                high_stop = True # trade hit stop loss
                # get stop loss timestamp
                high_sl_ts = sl_window.iloc[i:i+1].index[0]
                break 
        for i in range(len(sl_window)): 
            if sl_window.iloc[i] >= (df["Close"] + df["ADR"] * 0.25 ): #(stoploss_high + df["ATR"]):
                adr_stop = True # trade hit stop loss
                # get stop loss timestamp
                adr_sl_ts = sl_window.iloc[i:i+1].index[0] 
                break
        # if not stopped out then the stop window is signal:EOD
        # don't include trade if it is the last candle of the day
        if high_stop is False:
            if close_window.empty is False:
                high_sl_ts = close_window.iloc[-1:].index[0]
            else:
                high_sl_ts = START+TD
        if adr_stop is False:
            if close_window.empty is False:
                adr_sl_ts = close_window.iloc[-1:].index[0]
            else:
                adr_sl_ts = START+TD

        # Take profit price
        tp = df["Close"] - (df["ADR"] * 0.382)

        # High trade window
        high_tp_window = low[START+TD:high_sl_ts+TD]
        high_trade_start = START+TD
        high_trade_end = high_sl_ts
        high_target_pips = df["Close"] - tp
        high_sl_pips = df["Close"] - stoploss_high
        if high_tp_window.min() <= tp:
            high_win = 1
            high_gain = df["Close"] - tp
        else:
            high_loss = 1
            if high_stop is True:
                high_gain = high_sl_pips
            else:
                high_gain = df["Close"] - close_window.iloc[-1] if close_window.empty is False else 0           
        # ATR trade window
        adr_tp_window = low[START+TD:adr_sl_ts+TD]
        adr_trade_start = START+TD
        adr_trade_end = adr_sl_ts
        adr_target_pips = df["Close"] - tp
        adr_sl_pips = df["Close"] - (df["Close"] + df["ADR"] * 0.25) 
        if adr_tp_window.min() <= tp:
            adr_win = 1
            adr_gain = df["Close"] - tp
        else:
            adr_loss = 1
            if adr_stop is True:
                adr_gain = adr_sl_pips
            else:
                adr_gain = df["Close"] - close_window.iloc[-1] if close_window.empty is False else 0
    
    data = high_win, high_loss, \
        adr_win, adr_loss, \
        high_target_pips, adr_target_pips, \
        high_sl_pips, adr_sl_pips, \
        high_gain, adr_gain, \
        adr_trade_start, adr_trade_end, \
        high_trade_start, high_trade_end
    
    return data

# Set new columns 
pct_50_idr_cols = [
    "High_Win", "High_Loss", 
    "ADR_Win", "ADR_Loss", 
    "High_TP", "ADR_TP", 
    "High_SL", "ADR_SL", 
    "High_Gain", "ADR_Gain",
    "ADR_Trade_Start", "ADR_Trade_End",
    "High_Trade_Start", "High_Trade_End"
    ]

df[pct_50_idr_cols] = df.apply(get_bear_bbr_pip_gain, axis=1, args=[df["High"], df["Low"], df["Close"]], result_type='expand')


In [3542]:
# check signals

df[
    [
        "Open", "High", "Low","Close", "Range",
        "ATR", "Iday_Range", "ADR", "RSI", "RSI_DVG", "Close_%SMA",
        "Bear_BBR", "Bear_Engulf", "Shooting_Star",
        "Dark_Cloud", "Hammer","BBU_BO", "IHR", "S_R",
        "SMA32_Slope", "SMA_Trend", *pct_50_idr_cols
     ]
].query("Bear_BBR == True")

,Open,High,Low,Close,Range,ATR,Iday_Range,ADR,RSI,RSI_DVG,Close_%SMA,Bear_BBR,Bear_Engulf,Shooting_Star,Dark_Cloud,Hammer,BBU_BO,IHR,S_R,SMA32_Slope,SMA_Trend,High_Win,High_Loss,ADR_Win,ADR_Loss,High_TP,ADR_TP,High_SL,ADR_SL,High_Gain,ADR_Gain,ADR_Trade_Start,ADR_Trade_End,High_Trade_Start,High_Trade_End
Date,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
2025-04-24 04:15:00-04:00,1.330050,1.330100,1.329455,1.329755,0.000645,0.000923,0.005130,0.010477,65.919402,NaN,0.133426,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,32.602304,2,0.0,1.0,0.0,1.0,0.004002,0.004002,-0.000625,-0.002619,-0.000625,-0.002619,2025-04-24 04:30:00-04:00,2025-04-24 10:15:00-04:00,2025-04-24 04:30:00-04:00,2025-04-24 04:45:00-04:00
2025-04-24 05:15:00-04:00,1.331510,1.331515,1.330775,1.330800,0.000740,0.001015,0.006450,0.010477,66.652679,NaN,0.141492,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,31.269961,2,0.0,1.0,0.0,1.0,0.004002,0.004002,-0.000900,-0.002619,-0.000900,-0.002619,2025-04-24 05:30:00-04:00,2025-04-24 13:00:00-04:00,2025-04-24 05:30:00-04:00,2025-04-24 06:00:00-04:00
2025-04-24 10:45:00-04:00,1.332840,1.333270,1.330915,1.331455,0.002355,0.001099,0.008020,0.010477,56.168380,True,0.027703,True,True,NaN,NaN,NaN,NaN,True,4.0,43.389699,2,0.0,1.0,0.0,1.0,0.004002,0.004002,-0.001815,-0.002619,-0.001815,-0.002619,2025-04-24 11:00:00-04:00,2025-04-24 15:30:00-04:00,2025-04-24 11:00:00-04:00,2025-04-24 13:00:00-04:00
2025-04-24 15:45:00-04:00,1.334300,1.334425,1.333715,1.333740,0.000710,0.000722,0.009220,0.010477,62.063477,NaN,0.097659,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,30.787895,2,0.0,1.0,0.0,1.0,0.004002,0.004002,-0.000730,-0.002619,-0.000730,-0.000445,2025-04-24 16:00:00-04:00,2025-04-24 16:45:00-04:00,2025-04-24 16:00:00-04:00,2025-04-24 16:15:00-04:00
2025-04-25 03:15:00-04:00,1.330950,1.331350,1.330235,1.330305,0.001115,0.000969,0.006820,0.010625,51.626643,NaN,0.091534,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-27.205996,1,0.0,1.0,0.0,1.0,0.004059,0.004059,-0.001125,-0.002656,-0.001125,-0.002656,2025-04-25 03:30:00-04:00,2025-04-25 11:45:00-04:00,2025-04-25 03:30:00-04:00,2025-04-25 06:45:00-04:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-03-09 02:00:00-04:00,1.333365,1.334075,1.332485,1.332730,0.001590,0.001371,0.006865,0.009884,55.186083,NaN,0.149988,True,NaN,NaN,True,NaN,NaN,NaN,NaN,0.835504,2,0.0,1.0,0.0,1.0,0.003776,0.003776,-0.001345,-0.002471,-0.001345,-0.002471,2026-03-09 02:15:00-04:00,2026-03-09 05:30:00-04:00,2026-03-09 02:15:00-04:00,2026-03-09 03:30:00-04:00
2026-03-09 11:15:00-04:00,1.339725,1.339780,1.338540,1.339030,0.001240,0.001635,0.011760,0.009884,65.245502,NaN,0.182250,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,56.519957,2,0.0,1.0,0.0,1.0,0.003776,0.003776,-0.001040,-0.002471,-0.001040,-0.002471,2026-03-09 11:30:00-04:00,2026-03-09 15:15:00-04:00,2026-03-09 11:30:00-04:00,2026-03-09 12:00:00-04:00
2026-03-09 15:45:00-04:00,1.343725,1.344570,1.342880,1.342940,0.001690,0.001246,0.016260,0.009884,68.540948,True,0.230155,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,61.532578,2,0.0,1.0,0.0,1.0,0.003776,0.003776,-0.001630,-0.002471,-0.001630,-0.000450,2026-03-09 16:00:00-04:00,2026-03-09 16:45:00-04:00,2026-03-09 16:00:00-04:00,2026-03-09 16:45:00-04:00


In [3543]:
def prev_bbu_bo(s: Series, bbu_bo: Series):
    idx = int(s["Idx"])
    if idx > 0:
        return  bbu_bo.iloc[idx-1]
df["Prev_BBU_BO"] = df.apply(prev_bbu_bo, axis=1, args=[df["BBU_BO"]])

df2 = df.query("Bear_BBR == True ").between_time("01:00", "16:00")

In [3544]:
# Stats - High as SL
high_win_count = df2.query("Bear_BBR == True and High_Win > 0")["Bear_BBR"].count()
high_loss_count = df2.query("Bear_BBR == True and High_Loss > 0")["Bear_BBR"].count()
high_total_trades = high_win_count + high_loss_count
high_win_rate = high_win_count/high_total_trades * 100
high_win = df2.query("Bear_BBR == True and High_Gain > 0")["High_Gain"]
high_loss = df2.query("Bear_BBR == True and High_Gain < 0")["High_Gain"]
high_win_avg_pips = high_win.mean()
high_loss_avg_pips = high_loss.mean()
high_win_pips = high_win.sum()
high_loss_pips = high_loss.sum()
high_total_pips = high_win_pips + high_loss_pips

high_pct_50_idr = {
    "Win_Count": high_win_count,
    "Loss_Count": high_loss_count,
    "Total_Trades": high_total_trades,
    "Win_Rate": high_win_rate,
    "Avg_Win": high_win_avg_pips,
    "Avg_Loss": high_loss_avg_pips,
    "Win_Pips": high_win_pips,
    "Loss_Pips": high_loss_pips,
    "Total_Pips": high_total_pips
}

# Stats - High + ATR as SL
adr_win_count = df2.query("Bear_BBR == True and ADR_Win > 0")["Bear_BBR"].count()
adr_loss_count = df2.query("Bear_BBR == True and ADR_Loss > 0")["Bear_BBR"].count()
adr_total_trades = adr_win_count + adr_loss_count
adr_win_rate = adr_win_count/adr_total_trades * 100
adr_win = df2.query("Bear_BBR == True and ADR_Gain > 0")["ADR_Gain"]
adr_loss = df2.query("Bear_BBR == True and ADR_Gain < 0")["ADR_Gain"]
adr_win_avg_pips = adr_win.mean()
adr_loss_avg_pips = adr_loss.mean()
adr_win_pips = adr_win.sum()
adr_loss_pips = adr_loss.sum()
adr_total_pips = adr_win_pips + adr_loss_pips

adr_pct_50_idr = {
    "Win_Count": adr_win_count,
    "Loss_Count": adr_loss_count,
    "Total_Trades": adr_total_trades,
    "Win_Rate": adr_win_rate,
    "Avg_Win": adr_win_avg_pips,
    "Avg_Loss": adr_loss_avg_pips,
    "Win_Pips": adr_win_pips,
    "Loss_Pips": adr_loss_pips,
    "Total_Pips": adr_total_pips
}

pd.DataFrame([high_pct_50_idr, adr_pct_50_idr])


,Win_Count,Loss_Count,Total_Trades,Win_Rate,Avg_Win,Avg_Loss,Win_Pips,Loss_Pips,Total_Pips
0,50,199,249,20.080321,0.002788,-0.001018,0.186787,-0.185210,0.001577
1,79,170,249,31.726908,0.002785,-0.001995,0.303601,-0.279312,0.024289


In [3545]:
#pd.options.display.max_rows = 100
df2[["Prev_BBU_BO", "Iday_Range", "SMA32_Slope", "SMA16_Slope",*pct_50_idr_cols]].iloc[200:].query("High_Gain > 0")


,Prev_BBU_BO,Iday_Range,SMA32_Slope,SMA16_Slope,High_Win,High_Loss,ADR_Win,ADR_Loss,High_TP,ADR_TP,High_SL,ADR_SL,High_Gain,ADR_Gain,ADR_Trade_Start,ADR_Trade_End,High_Trade_Start,High_Trade_End
Date,,,,,,,,,,,,,,,,,,
2026-01-09 08:45:00-05:00,True,0.005230,12.053992,5.205098e+01,1.0,0.0,1.0,0.0,0.002812,0.002812,-0.001545,-0.001840,0.002812,0.002812,2026-01-09 09:00:00-05:00,2026-01-09 16:45:00-05:00,2026-01-09 09:00:00-05:00,2026-01-09 16:45:00-05:00
2026-01-14 09:45:00-05:00,True,0.004430,17.869105,1.358599e+01,1.0,0.0,1.0,0.0,0.002804,0.002804,-0.000735,-0.001835,0.002804,0.002804,2026-01-14 10:00:00-05:00,2026-01-14 16:45:00-05:00,2026-01-14 10:00:00-05:00,2026-01-14 16:45:00-05:00
2026-01-20 03:15:00-05:00,True,0.008140,55.436546,6.216358e+01,1.0,0.0,1.0,0.0,0.002727,0.002727,-0.001250,-0.001785,0.002727,0.002727,2026-01-20 03:30:00-05:00,2026-01-20 16:45:00-05:00,2026-01-20 03:30:00-05:00,2026-01-20 16:45:00-05:00
2026-01-20 10:45:00-05:00,NaN,0.008140,-12.850691,-4.240740e-11,1.0,0.0,0.0,1.0,0.002727,0.002727,-0.002030,-0.001785,0.002727,-0.001785,2026-01-20 11:00:00-05:00,2026-01-20 12:00:00-05:00,2026-01-20 11:00:00-05:00,2026-01-20 16:45:00-05:00
2026-01-21 02:30:00-05:00,True,0.002695,2.743324,1.713627e+01,1.0,0.0,1.0,0.0,0.002771,0.002771,-0.000975,-0.001814,0.002771,0.002771,2026-01-21 02:45:00-05:00,2026-01-21 10:00:00-05:00,2026-01-21 02:45:00-05:00,2026-01-21 09:30:00-05:00
2026-01-28 02:15:00-05:00,True,0.006205,-10.070955,2.357926e+01,1.0,0.0,1.0,0.0,0.003101,0.003101,-0.001985,-0.002029,0.003101,0.003101,2026-01-28 02:30:00-05:00,2026-01-28 16:45:00-05:00,2026-01-28 02:30:00-05:00,2026-01-28 16:45:00-05:00
2026-01-28 09:45:00-05:00,NaN,0.008610,-16.315193,-8.951737e-01,1.0,0.0,1.0,0.0,0.003101,0.003101,-0.001495,-0.002029,0.003101,0.003101,2026-01-28 10:00:00-05:00,2026-01-28 16:00:00-05:00,2026-01-28 10:00:00-05:00,2026-01-28 15:30:00-05:00
2026-01-29 03:15:00-05:00,NaN,0.005480,18.192893,2.496786e+01,1.0,0.0,1.0,0.0,0.003171,0.003171,-0.000795,-0.002075,0.003171,0.003171,2026-01-29 03:30:00-05:00,2026-01-29 16:45:00-05:00,2026-01-29 03:30:00-05:00,2026-01-29 16:45:00-05:00
2026-01-30 03:15:00-05:00,NaN,0.008320,-34.142177,2.477136e+01,1.0,0.0,1.0,0.0,0.003177,0.003177,-0.001460,-0.002079,0.003177,0.003177,2026-01-30 03:30:00-05:00,2026-01-30 07:00:00-05:00,2026-01-30 03:30:00-05:00,2026-01-30 07:00:00-05:00


In [2043]:
df.loc["2024-12-06 08:45:00-05:00"]

Open                 1.280485
High                  1.28064
Low                  1.278535
Close                 1.27948
Idx                     18302
Body                 0.001005
Range                0.002105
UWick                0.000155
LWick                0.000945
Close_%High          0.551069
Open_%High           0.073634
Iday_Idx                   62
Iday_High            1.281145
Iday_Low             1.273985
Iday_Range            0.00716
Close_%DHigh         0.232542
Open_%DHigh          0.092179
Yday_High             1.27708
Yday_Low             1.269315
Day_Idx                   192
ADR                  0.009483
ATR                  0.001106
SMA4                 1.278546
SMA16                1.276926
SMA32                1.276297
BB_Upper_16_2        1.279555
BB_Lower_16_2        1.274297
Close_%SMA           0.199992
RSI                 69.540125
RSI_DVG                   NaN
Sig_High             1.277425
Sig_Low              1.273985
SMA_Trend                   2
SMA4_Slope